In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze Ingestion
# MAGIC Reads all 17 landing sources from the raw Volume via Auto Loader
# MAGIC (`trigger(availableNow=True)` — one-shot backfill, not continuous streaming)
# MAGIC and writes each to its own Delta table in the `bronze` schema.
# MAGIC
# MAGIC Depends on `verify_bronze` (Python script task) having already created
# MAGIC the `bronze` schema — this notebook does not create catalog/schema objects.

# COMMAND ----------

dbutils.widgets.text("catalog", "credit_risk_fraud_detection")
CATALOG = dbutils.widgets.get("catalog")

VOLUME_PATH = f"/Volumes/{CATALOG}/landing/raw_volume"
BRONZE_SCHEMA = f"{CATALOG}.bronze"

print(f"Catalog:      {CATALOG}")
print(f"Volume path:  {VOLUME_PATH}")
print(f"Bronze schema: {BRONZE_SCHEMA}")
# COMMAND ----------

def ingest_bronze(
    entity: str,
    source_subpath: str,
    path_glob_filter: str = "*.parquet",
) -> None:
    """
    Reads parquet files for one entity via Auto Loader and writes to
    {BRONZE_SCHEMA}.{entity}. Blocks until the one-shot backfill completes
    (trigger=availableNow), so it's safe to call these sequentially in a loop.
    """
    source_path = f"{VOLUME_PATH}/{source_subpath}"
    checkpoint_path = f"{VOLUME_PATH}/_checkpoints/{entity}"
    schema_location = f"{VOLUME_PATH}/_schemas/{entity}"
    target_table = f"{BRONZE_SCHEMA}.{entity}"

    print(f"Ingesting {entity:<24} <- {source_path}  (filter: {path_glob_filter})")

    stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("pathGlobFilter", path_glob_filter)
        .option("rescuedDataColumn", "_rescued_data")
        .load(source_path)
        .writeStream
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )
    stream.awaitTermination()

    count = spark.table(target_table).count()
    print(f"  -> {target_table}: {count:,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Standard entities — one landing folder, one Bronze table each

# COMMAND ----------

STANDARD_ENTITIES = [
    "loan_applications",
    "bureau_pulls",
    "underwriting_decisions",
    "disbursements",
    "repayment_schedules",
    "repayment_events",
    "nach_bounces",
    "transactions",
    "loan_personal_detail",
    "loan_home_detail",
    "loan_auto_detail",
    "credit_card_detail",
]

for entity in STANDARD_ENTITIES:
    ingest_bronze(entity, source_subpath=entity)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Customer entities — special case
# MAGIC `write_customers()` writes customer, KYC, and credit-profile records into
# MAGIC the SAME `customers/` folder, distinguished only by filename suffix:
# MAGIC - base customer rows: `customers_YYYYMMDD_HHMMSS_NNN.parquet`
# MAGIC - KYC profiles:       `customers_YYYYMMDD_HHMMSS_NNN_kyc.parquet`
# MAGIC - credit profiles:    `customers_YYYYMMDD_HHMMSS_NNN_credit.parquet`
# MAGIC
# MAGIC Base filenames end in a digit before `.parquet`; the suffixed ones end in
# MAGIC a letter (`c`) before `.parquet` — that difference is enough for a glob
# MAGIC filter to split all three into separate Bronze tables from one folder.

# COMMAND ----------

ingest_bronze("customers",       source_subpath="customers", path_glob_filter="customers_*[0-9].parquet")
ingest_bronze("kyc_profiles",    source_subpath="customers", path_glob_filter="*_kyc.parquet")
ingest_bronze("credit_profiles", source_subpath="customers", path_glob_filter="*_credit.parquet")

# COMMAND ----------

# MAGIC %md
# MAGIC ## CDC streams — one directory level deeper (`cdc/loan_status`, `cdc/customer_updates`)

# COMMAND ----------

ingest_bronze("cdc_loan_status",      source_subpath="cdc/loan_status")
ingest_bronze("cdc_customer_updates", source_subpath="cdc/customer_updates")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Summary

# COMMAND ----------

bronze_tables = [row.tableName for row in spark.sql(f"SHOW TABLES IN {BRONZE_SCHEMA}").collect()]
print(f"Bronze tables in {BRONZE_SCHEMA}: {len(bronze_tables)}")
for t in sorted(bronze_tables):
    n = spark.table(f"{BRONZE_SCHEMA}.{t}").count()
    print(f"  {t:<24} {n:>12,} rows")